# Regional monthly cycles

Multi-year mean annual cycles of surface biogeochemical and physical fields averaged over eight ocean regions, model and observations.

Model fields come from the HAMOCC monthly history files (`hbgcm`); SST, SSS and mixed layer depth come from the BLOM monthly files (`hm`). They are interpolated to a 1x1 degree grid, and the region means are area-weighted (cos latitude). Regions are the basin masks used for the zonal means restricted to the latitude bands of the legacy diagnostics.

In [ ]:
import matplotlib.pyplot as plt
import yaml
from pyprojroot import here

from blomster import grid
from hamocc import climatology, io, obs, plots, regions, variables, zonal

In [ ]:
region_names = list(variables.REGIONS)
compare_obs = True

In [ ]:
with open(here('notebooks_hamocc/settings.yml')) as f:
    cfg = yaml.safe_load(f)
exp = cfg['experiment']
dsets = cfg['datasets']
zcfg = dsets['zonal']
ys, ye = exp['year_start'], exp['year_end']

lat, lon, _, _, _, _ = grid.load(cfg['grids'][exp['grid']]['grid_file'])
regridder = zonal.make_regridder(lat, lon)
target = zonal.target_grid()

basin_masks = {b: zonal.load_basin_mask(f"{zcfg['basin_mask_root']}/{zcfg['basin_mask_file'].format(basin=b)}",
                                         zcfg['basin_mask_var']) for b in set(variables.REGION_BASINS.values())}
masks = {r: regions.region_mask(basin_masks, r, target) for r in region_names}

bgc_names = [n for n in variables.REGIONAL_VARS if n not in variables.PHYS_MONTHLY_VARS]
bgc_needed = sorted({h for n in bgc_names for h in climatology.history_names(n, monthly=True)})
ds_bgc = io.open_history(io.history_files(exp['input_root'], exp['caseid'], 'hbgcm', ys, ye), bgc_needed)
ds_phys = io.open_history(io.history_files(exp['input_root'], exp['caseid'], 'hm', ys, ye),
                          cfg['variables']['hm']).rename(mldb04='mld')

def model_cycles(name):
    ds = ds_phys if name in variables.PHYS_MONTHLY_VARS else ds_bgc
    cycle = climatology.monthly_cycle(ds, name).load()
    on_1x1 = regridder(cycle, skipna=True, na_thres=1.0)
    on_1x1.attrs.update(cycle.attrs)
    return {r: regions.model_cycle(on_1x1, masks[r]) for r in region_names}

def obs_cycles(name):
    spec = dsets['regional'].get(name)
    if not compare_obs or spec is None:
        return None
    return {r: regions.obs_cycle(obs.load_obs_region(spec, dsets['obs_root'], r), masks[r])
            for r in region_names}

def show(name):
    plots.cycle_panels(model_cycles(name), obs_cycles(name))
    plt.show()

```{note}
Observation file names and scale factors (`datasets.regional` and `datasets.units_verified` in `recipe_hamocc.yml`) and the basin mask file are not yet verified. If an observation file already holds a regional mean it is used as is; if it holds a lat/lon field it is averaged over the same region mask as the model.
```

## Carbon cycle

In [ ]:
for name in ['pco2', 'co2fxn', 'pp_tot', 'ppint']:
    show(name)

## Surface nutrients and oxygen

In [ ]:
for name in ['srfo2', 'srfsi', 'srfpo4', 'srfno3']:
    show(name)

## Physical surface fields

In [ ]:
for name in ['sst', 'sss', 'mld']:
    show(name)